# Chapter 3 — Stacks and Queues

- **Stack**: LIFO. `push`, `pop`, `peek`, `is_empty` in `O(1)`. In Python, `list.append` / `list.pop`.
- **Queue**: FIFO. `add`, `remove`, `peek`, `is_empty`. In Python use `collections.deque` (`list.pop(0)` is `O(N)`!).

We implement both on top of a linked list first.

In [1]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from typing import Generic, TypeVar

T = TypeVar("T")


@dataclass
class _Node(Generic[T]):
    val: T
    next: _Node[T] | None = None


class Stack(Generic[T]):
    def __init__(self) -> None:
        self._top: _Node[T] | None = None
        self._size = 0

    def push(self, item: T) -> None:
        self._top = _Node(item, self._top)
        self._size += 1

    def pop(self) -> T:
        if self._top is None:
            raise IndexError("pop from empty stack")
        item, self._top = self._top.val, self._top.next
        self._size -= 1
        return item

    def peek(self) -> T:
        if self._top is None:
            raise IndexError("peek at empty stack")
        return self._top.val

    def is_empty(self) -> bool:
        return self._top is None

    def __len__(self) -> int:
        return self._size


class Queue(Generic[T]):
    def __init__(self) -> None:
        self._first: _Node[T] | None = None
        self._last: _Node[T] | None = None

    def add(self, item: T) -> None:
        node = _Node(item)
        if self._last:
            self._last.next = node
        self._last = node
        if self._first is None:
            self._first = node

    def remove(self) -> T:
        if self._first is None:
            raise IndexError("remove from empty queue")
        item, self._first = self._first.val, self._first.next
        if self._first is None:
            self._last = None
        return item

    def peek(self) -> T:
        if self._first is None:
            raise IndexError("peek at empty queue")
        return self._first.val

    def is_empty(self) -> bool:
        return self._first is None


s: Stack[int] = Stack()
q: Queue[int] = Queue()
for i in range(3):
    s.push(i)
    q.add(i)
assert [s.pop() for _ in range(3)] == [2, 1, 0]
assert [q.remove() for _ in range(3)] == [0, 1, 2]
print("stack/queue ok")

stack/queue ok


## 3.1 Three in One
Implement three stacks using a single array. Fixed division is simplest; below is the **flexible** variant where stacks shift neighbours to make room (the array is treated as circular).

In [2]:
class FixedMultiStack:
    def __init__(self, stack_size: int, num_stacks: int = 3) -> None:
        self.cap = stack_size
        self.values: list[object] = [None] * (stack_size * num_stacks)
        self.sizes = [0] * num_stacks

    def push(self, stack: int, value: object) -> None:
        if self.sizes[stack] == self.cap:
            raise OverflowError(f"stack {stack} full")
        self.values[stack * self.cap + self.sizes[stack]] = value
        self.sizes[stack] += 1

    def pop(self, stack: int) -> object:
        if self.sizes[stack] == 0:
            raise IndexError(f"stack {stack} empty")
        self.sizes[stack] -= 1
        idx = stack * self.cap + self.sizes[stack]
        value, self.values[idx] = self.values[idx], None
        return value

    def peek(self, stack: int) -> object:
        if self.sizes[stack] == 0:
            raise IndexError(f"stack {stack} empty")
        return self.values[stack * self.cap + self.sizes[stack] - 1]


class FlexibleMultiStack:
    """Stacks start with equal slices but grow by shifting the next stack over."""

    @dataclass
    class Info:
        start: int
        size: int
        capacity: int

    def __init__(self, num_stacks: int, default_size: int) -> None:
        self.info = [self.Info(i * default_size, 0, default_size) for i in range(num_stacks)]
        self.values: list[object] = [None] * (num_stacks * default_size)

    def _adjust(self, i: int) -> int:
        return i % len(self.values)

    def _top_index(self, s: int) -> int:
        inf = self.info[s]
        return self._adjust(inf.start + inf.size - 1)

    def _free_slots(self) -> int:
        return len(self.values) - sum(i.size for i in self.info)

    def push(self, s: int, value: object) -> None:
        if self._free_slots() == 0:
            raise OverflowError("all stacks full")
        if self.info[s].size >= self.info[s].capacity:
            self._expand(s)
        self.info[s].size += 1
        self.values[self._top_index(s)] = value

    def pop(self, s: int) -> object:
        inf = self.info[s]
        if inf.size == 0:
            raise IndexError(f"stack {s} empty")
        idx = self._top_index(s)
        value, self.values[idx] = self.values[idx], None
        inf.size -= 1
        return value

    def peek(self, s: int) -> object:
        return self.values[self._top_index(s)]

    def _shift(self, s: int) -> None:
        """Shift stack s right by one, shifting the next stack first if s is at capacity."""
        inf = self.info[s]
        if inf.size >= inf.capacity:
            self._shift((s + 1) % len(self.info))
            inf.capacity += 1
        idx = self._adjust(inf.start + inf.capacity - 1)
        while idx != self._adjust(inf.start):
            prev = self._adjust(idx - 1)
            self.values[idx] = self.values[prev]
            idx = prev
        self.values[self._adjust(inf.start)] = None
        inf.start = self._adjust(inf.start + 1)
        inf.capacity -= 1

    def _expand(self, s: int) -> None:
        self._shift((s + 1) % len(self.info))
        self.info[s].capacity += 1


fm = FixedMultiStack(2)
fm.push(0, "a"); fm.push(2, "z")
assert fm.pop(2) == "z" and fm.peek(0) == "a"

ms = FlexibleMultiStack(3, 2)
for v in range(5):
    ms.push(0, v)       # stack 0 grows past its initial 2 slots
ms.push(1, "b")
assert ms._free_slots() == 0
assert [ms.pop(0) for _ in range(5)] == [4, 3, 2, 1, 0]
assert ms.pop(1) == "b"
print("3.1 ok")

3.1 ok


## 3.2 Stack Min
A stack that also reports its minimum in `O(1)`. Store, alongside the main stack, a second stack of running minimums (only pushed when a new min appears, to save space).

In [3]:
class MinStack:
    def __init__(self) -> None:
        self._items: list[int] = []
        self._mins: list[int] = []

    def push(self, v: int) -> None:
        self._items.append(v)
        if not self._mins or v <= self._mins[-1]:
            self._mins.append(v)

    def pop(self) -> int:
        v = self._items.pop()
        if v == self._mins[-1]:
            self._mins.pop()
        return v

    def min(self) -> int:
        return self._mins[-1]


st = MinStack()
for v in [5, 6, 3, 7, 3, 1]:
    st.push(v)
mins = []
while st._items:
    mins.append(st.min())
    st.pop()
assert mins == [1, 3, 3, 3, 5, 5]
print("3.2 ok")

3.2 ok


## 3.3 Stack of Plates
`SetOfStacks` starts a new stack when the current one reaches capacity; `push`/`pop` act like a single stack. Follow-up: `pop_at(index)` pops from a specific sub-stack (here we "roll over" so every sub-stack but the last stays full).

In [4]:
class SetOfStacks:
    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.stacks: list[deque[int]] = []

    def push(self, v: int) -> None:
        if not self.stacks or len(self.stacks[-1]) == self.capacity:
            self.stacks.append(deque())
        self.stacks[-1].append(v)

    def pop(self) -> int:
        return self.pop_at(len(self.stacks) - 1)

    def pop_at(self, index: int) -> int:
        if not self.stacks:
            raise IndexError("empty")
        v = self.stacks[index].pop()
        # shift the bottom of each later stack onto the previous one
        for i in range(index, len(self.stacks) - 1):
            self.stacks[i].append(self.stacks[i + 1].popleft())
        if not self.stacks[-1]:
            self.stacks.pop()
        return v


plates = SetOfStacks(3)
for v in range(1, 8):
    plates.push(v)
assert [list(s) for s in plates.stacks] == [[1, 2, 3], [4, 5, 6], [7]]
assert plates.pop_at(0) == 3
assert [list(s) for s in plates.stacks] == [[1, 2, 4], [5, 6, 7]]
assert [plates.pop() for _ in range(6)] == [7, 6, 5, 4, 2, 1]
print("3.3 ok")

3.3 ok


## 3.4 Queue via Stacks
Use two stacks: push onto `newest`; when removing, if `oldest` is empty, pour `newest` into it (reversing order). Each element is moved at most once → `O(1)` amortized.

In [5]:
class MyQueue(Generic[T]):
    def __init__(self) -> None:
        self._newest: list[T] = []
        self._oldest: list[T] = []

    def add(self, v: T) -> None:
        self._newest.append(v)

    def _shift(self) -> None:
        if not self._oldest:
            while self._newest:
                self._oldest.append(self._newest.pop())

    def peek(self) -> T:
        self._shift()
        return self._oldest[-1]

    def remove(self) -> T:
        self._shift()
        return self._oldest.pop()

    def __len__(self) -> int:
        return len(self._newest) + len(self._oldest)


mq: MyQueue[int] = MyQueue()
for v in range(3):
    mq.add(v)
assert mq.remove() == 0
mq.add(3)
assert [mq.remove() for _ in range(3)] == [1, 2, 3]
print("3.4 ok")

3.4 ok


## 3.5 Sort Stack
Sort a stack so the smallest item is on top, using only one extra stack. Insertion-sort style: pop from `s`, and while the buffer's top is bigger, move it back to `s`. O(N²) time, O(N) space.

In [6]:
def sort_stack(s: list[int]) -> list[int]:
    r: list[int] = []  # buffer kept sorted, largest on top
    while s:
        tmp = s.pop()
        while r and r[-1] > tmp:
            s.append(r.pop())
        r.append(tmp)
    # r has the largest on top; move back so the smallest is on top of s
    while r:
        s.append(r.pop())
    return s


st = [5, 1, 4, 2, 3]
sort_stack(st)
assert st == [5, 4, 3, 2, 1]  # top (end of list) is the smallest
print("3.5 ok")

3.5 ok


## 3.6 Animal Shelter
FIFO shelter for dogs and cats. `dequeue_any` returns the oldest animal; `dequeue_dog`/`dequeue_cat` return the oldest of that type. Keep separate queues and stamp each animal with an arrival order.

In [7]:
from itertools import count


@dataclass
class Animal:
    name: str
    order: int = 0


class Dog(Animal): pass
class Cat(Animal): pass


class AnimalShelter:
    def __init__(self) -> None:
        self._dogs: deque[Dog] = deque()
        self._cats: deque[Cat] = deque()
        self._clock = count()

    def enqueue(self, animal: Animal) -> None:
        animal.order = next(self._clock)
        match animal:
            case Dog():
                self._dogs.append(animal)
            case Cat():
                self._cats.append(animal)
            case _:
                raise TypeError(animal)

    def dequeue_any(self) -> Animal:
        if not self._dogs:
            return self.dequeue_cat()
        if not self._cats:
            return self.dequeue_dog()
        return self.dequeue_dog() if self._dogs[0].order < self._cats[0].order else self.dequeue_cat()

    def dequeue_dog(self) -> Dog:
        return self._dogs.popleft()

    def dequeue_cat(self) -> Cat:
        return self._cats.popleft()


shelter = AnimalShelter()
for a in [Dog("rex"), Cat("tom"), Dog("fido"), Cat("kitty")]:
    shelter.enqueue(a)
assert shelter.dequeue_cat().name == "tom"
assert shelter.dequeue_any().name == "rex"
assert shelter.dequeue_any().name == "fido"
assert shelter.dequeue_any().name == "kitty"
print("3.6 ok")

3.6 ok
